# 第 11 章：流式输出与服务化

上一章解决了“修改后还能不能信”。这一章把 Agent 放进持续对话和 HTTP 服务，解决三个问题：怎样边生成边显示，怎样限制一次请求的资源消耗，以及怎样让服务重启后仍能找回会话。

| 读者契约 | 内容 |
|---|---|
| 你已会 | Python async、Runner/Session、工具调用与第 10 章评估 |
| 读完后你能 | 正确消费流式事件；限制调用循环；观察上下文压缩；说明服务的存储与身份边界 |
| 本章不碰 | 多实例部署、自动扩缩容、分布式限流 |
| 主线 | 离线事件实验 → 预算失败 → 压缩实验 → HTTP；不需要 API key |
| 深读 | 真实模型流式、Docker 与生产边界 |

**先猜：** 收到两段 `订单`、`已签收`，最后又收到 `订单已签收`，应该显示几遍？先保留你的答案，再运行实验。

In [1]:
from pathlib import Path
import sys

course_dir = Path.cwd() if (Path.cwd() / "runtime_example").is_dir() else Path.cwd() / "google-adk"
assert course_dir.is_dir(), "请从仓库根目录或 google-adk 目录运行 notebook"
if str(course_dir) not in sys.path:
    sys.path.insert(0, str(course_dir))
from importlib.metadata import version
assert version("google-adk") == "2.6.3"

## 1. 增量是预览，完整事件是这一轮结果

ADK 2.6.3 的流式模型接口可以先产生 `partial=True` 的文本增量，再产生 `partial=False` 的完整聚合内容。把两者都追加到字符串，页面就会显示两遍。

要直接通过 Runner 观察流式，需显式传入 `RunConfig(streaming_mode=StreamingMode.SSE)`；HTTP 请求里的 `streaming: true` 在服务端映射到同一配置。下面的 `verify()` 已经这样设置。

还要区分三个层次：一个 Event 可以装多个 Part；一个文本 Part 可以有多个字符；字符数和 token 数又不同。因此“收到 10 个事件”不能证明“生成了 10 个 token”。

本例让模型替身固定发送两段文本，**Runner、Event、工具调用处理都是真实 ADK**。这样可以稳定复现重复显示，而不依赖供应商恰好怎样分块。

In [2]:
from runtime_example.verify import TextCollector, verify

await verify()

Skipping missing token usage metadata for agent answer and model offline-runtime
D:\Python\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()
D:\Projects\PythonProjects\agent-frameworks-learning\google-adk\runtime_example\verify.py:130: UserWarning: [EXPERIMENTAL] BaseEventsSummarizer: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  overlap_size=0, summarizer=FixedSummary()))
D:\Projects\PythonProjects\agent-frameworks-learning\google-adk\runtime_example\verify.py:129: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(compaction_interval=2,
Skipping missing token usage metadat

PASS 流式：增量预览与最终结果分开处理，没有重复拼接
PASS 预算：两次模型调用后停止，不能把循环成功误当业务成功
PASS 压缩：摘要事件存在，后续模型请求实际收到摘要；业务账本仍由工具读取


预期看到三项 PASS：流式文本不重复、调用预算中止循环、压缩摘要进入后续请求。任何断言失败都会停止执行。

`TextCollector` 的核心规则如下。实际项目还应按 invocation/请求划分 collector，避免把两轮对话混在一起；多 Agent 场景要明确选择展示哪个 Agent 的输出。`partial=False` 本身不等于根 Agent 的最终回答。

In [3]:
import inspect
print(inspect.getsource(TextCollector))

@dataclass
class TextCollector:
    """只收指定 Agent 文本；最终聚合结果替换预览，不再次追加。"""

    author: str
    preview: str = ""
    final: str = ""
    partial_events: int = 0

    def accept(self, event: Event):
        if event.author != self.author or not event.content:
            return
        text = "".join(p.text or "" for p in event.content.parts or [] if not p.thought)
        if event.partial:
            self.preview += text
            self.partial_events += 1
        elif event.is_final_response():
            self.final = text

    @property
    def text(self):
        return self.final or self.preview



## 2. 四种预算要分别设置

| 预算 | 放在哪里 | 它不保证什么 |
|---|---|---|
| 输入长度 | HTTP 请求校验与检索结果截断 | 字符数不是精确 token 数 |
| 单次输出 token | 模型配置，例如 LiteLlm 的 `max_tokens` | 不限制整个工具循环 |
| 一次 invocation 的模型调用数 | `RunConfig(max_llm_calls=...)` | 不限制跨请求累计成本，也不等于供应商重试次数 |
| 时间与并发 | 服务超时、队列、信号量 | 超时不保证已经发生的外部副作用被撤销 |

实验中的 `DemoModel(repeat_tool=True)` 每次都请求 `ping`。预算设为 2 后，第三次模型调用前抛出 `LlmCallsLimitExceededError`。这是一条必须处理的失败路径：返回“任务未完成”，记录原因，不能把最后一个工具输出冒充整项任务成功。

In [4]:
from google.adk.agents.run_config import RunConfig

request_budget = RunConfig(max_llm_calls=4)
assert request_budget.max_llm_calls == 4
# 将它传给 runner.run_async(..., run_config=request_budget)。
# 2.6.3 中 <= 0 表示不限制；生产配置不要误把 0 当作“禁止调用”。

## 3. 对话越长，先问“什么需要留下”

假设售后会话聊了 50 轮：订单号和用户偏好需要继续可见；十轮前的措辞可以摘要；退款审批和已执行的交易必须保存在业务数据库。它们不能只留在可能失真的自然语言摘要里。

`EventsCompactionConfig` 控制会话事件进入后续模型上下文时的压缩。下面采用每两次 invocation 压缩、无重叠的配置。前面的实验不仅检查配置对象，还断言 Session 出现 compaction 事件、后续 LLM request 实际含有摘要。

> ⚠️ `EventsCompactionConfig` 与 `BaseEventsSummarizer` 在 2.6.3 标记为 experimental，接口可能变化；升级 ADK 后先跑本节的契约测试。

In [5]:
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.agents import Agent
from runtime_example.verify import DemoModel, FixedSummary

compact_app = App(
    name="compact_demo",
    root_agent=Agent(name="answer", model=DemoModel()),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=2, overlap_size=0, summarizer=FixedSummary(),
    ),
)
print(compact_app.events_compaction_config.model_dump(exclude={"summarizer"}))

{'compaction_interval': 2, 'overlap_size': 0, 'token_threshold': None, 'event_retention_size': None}


C:\Users\37245\AppData\Local\Temp\ipykernel_35700\2940277482.py:9: UserWarning: [EXPERIMENTAL] BaseEventsSummarizer: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  compaction_interval=2, overlap_size=0, summarizer=FixedSummary(),
C:\Users\37245\AppData\Local\Temp\ipykernel_35700\2940277482.py:8: UserWarning: [EXPERIMENTAL] EventsCompactionConfig: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  events_compaction_config=EventsCompactionConfig(


这里的 `FixedSummary` 是教学替身，只用于验证接线。换成真实摘要模型后，必须用第 10 章的方法测量：关键事实保留率、错误事实、后续任务成功率和摘要新增成本。压缩不是自动正确的“无限记忆”。

<details><summary>深入：按 token 触发与数据保留</summary>

2.6.3 也支持成对配置 `token_threshold` 和 `event_retention_size`。触发依赖最近一次可用的 prompt token 用量，并发生在调用之后；供应商用量缺失时，不能把它当成可靠的硬窗口限制。压缩改变模型接收的历史表示，不等于已删除数据库中的原始会话数据。过期、删除与合规保留需要另设策略。

</details>

## 4. 从 Runner 到 HTTP

先在终端执行下面的离线验收。**工作目录为仓库根目录**；脚本启动临时 `adk api_server`，创建会话、请求 `/run_sse`，再关闭服务。

```bash
uv run python google-adk/practice/c11_1.py
# 可选：真正调用 DeepSeek，验证供应商的分块行为
uv run --env-file .env python google-adk/practice/c11_1.py --backend deepseek
```

`/run` 返回事件列表；`/run_sse` 逐个推送事件。请求中的 `streaming: true` 启用模型流式，否则 SSE 传输本身也可能只推送完整事件。客户端断言最终文本存在、来源正确；不硬编码供应商必须分多少块。

手动启动标准工程：

```bash
uv run --env-file .env adk api_server google-adk --host 127.0.0.1 --port 8000 --session_service_uri memory://
```

显式 `memory://` 用于一次性调试。ADK 2.6.3 CLI 在可写的本地工程中默认会使用 `.adk` 下的本地存储，不能笼统认为 CLI 默认一定是内存。需要持久会话时指定 SQLite URI，并记录数据库位置；同时配置 Artifact 等独立存储。Session 数据库不能替代退款账本。

## 5. 服务入口的责任

`user_id` 是会话查找参数，不是登录证明。前端不能通过随便填写 `user_id` 获得其他人的数据。下一章会在 HTTP 边界认证身份，再把可信身份绑定到工具。

<details><summary>深入：Docker 最小运行</summary>

在仓库根目录构建，镜像使用锁文件，运行时只注入需要的密钥，不把 `.env` 打进镜像：

```bash
docker build -f google-adk/Dockerfile -t adk-course-demo .
docker run --rm -p 127.0.0.1:8000:8000 --env-file .env adk-course-demo
```

这个镜像用于本机实验。正式部署还需要外部身份认证、持久卷/数据库、超时与限流、密钥托管、日志脱敏和回滚流程。第 15 章交付可重复验收的单机应用；它不是已经完成这些运维工作的公网产品。

</details>

## 🧪 本章练习

1. 在 collector 中插入另一个 Agent 的文本事件，证明最终回答没被污染。
2. 把预算从 2 改成 1，先预测已发生的工具调用数，再检查事件；说明预算终止为什么不等于事务回滚。
3. 写三轮对话：订单号、偏好、闲聊。比较压缩前后的模型请求，列出必须保留的事实和不应写进摘要的秘密。

参考实现：[runtime_example](runtime_example/verify.py)、[HTTP 流式练习](practice/c11_1.py)。

## 用五句话复述

增量与最终结果有什么区别？Event 为什么不等于 token？调用次数预算保护哪一段？摘要会丢失什么？服务端怎样知道是谁在调用？

下一章：[12-安全与鉴权](12-安全与鉴权.ipynb)。

核对资料：[ADK 事件](https://adk.dev/events/)、[上下文压缩](https://adk.dev/context/compaction/)。本章代码锁定 ADK 2.6.3。